In [1]:
import os
import gymnasium as gym
import panda_gym
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
from stable_baselines3.common.monitor import Monitor

# 1. 环境配置
env_id = "PandaPickAndPlace-v3"  # 抓取并放置任务
log_dir = "./logs/sac_panda_results/"
os.makedirs(log_dir, exist_ok=True)

def make_env():
    env = gym.make(env_id, render_mode="rgb_array") # 训练时不需要渲染
    env = Monitor(env)  # 记录每个回合的奖励和长度
    return env

# 使用向量化环境并进行归一化（具身智能的关键：对状态和奖励进行归一化）
env = DummyVecEnv([make_env])
env = VecNormalize(env, norm_obs=True, norm_reward=True, clip_obs=10.)

# 2. 评估回调：这是判断算法是否收敛的核心工具
# 它会定期在一个独立环境里测试 N 个回合，计算“成功率”
eval_env = DummyVecEnv([make_env])
eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False, training=False) # 评估环境不训练归一化参数

eval_callback = EvalCallback(
    eval_env, 
    best_model_save_path=log_dir,
    log_path=log_dir, 
    eval_freq=5000, # 每 5000 步评估一次
    deterministic=True, 
    render=False
)

# 每 20000 步保存一个快照
checkpoint_callback = CheckpointCallback(save_freq=20000, save_path=log_dir, name_prefix="sac_model")

# 3. 定义 SAC 模型
# 针对机械臂任务微调过的超参数
policy_kwargs = dict(net_arch=[256, 256, 256]) # 增加网络深度处理复杂接触

model = SAC(
    "MultiInputPolicy", # 因为环境输入包含图像/目标/状态，是一个 Dict
    env,
    learning_rate=1e-3,
    buffer_size=1000000,
    batch_size=256,
    tau=0.005,
    gamma=0.95, # 机械臂任务通常关注短期反馈
    learning_starts=1000,
    policy_kwargs=policy_kwargs,
    tensorboard_log=log_dir,
    verbose=1,
    device="cuda" # 如果有显卡
)

# 4. 开始训练
print("开始训练...")
model.learn(
    total_timesteps=200000, 
    callback=[eval_callback, checkpoint_callback],
    tb_log_name="SAC_Panda_PickPlace"
)

# 5. 保存最终模型和归一化统计量（非常重要！）
model.save("sac_panda_final")
env.save("vec_normalize.pkl")

pybullet build time: Oct 21 2025 17:40:50


argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886
argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886
Using cpu device
开始训练...
Logging to ./logs/sac_panda_results/SAC_Panda_PickPlace_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -50      |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 4        |
|    fps             | 1340     |
|    time_elapsed    | 0        |
|    total_timesteps | 200      |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 50       |
|    ep_rew_mean     | -50      |
|    success_rate    | 0        |
| time/              |          |
|    episodes        | 8       

KeyboardInterrupt: 

In [1]:
import gymnasium as gym
import panda_gym
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

# 1. 环境配置 (必须与训练时一致)
env_id = "PandaPickAndPlace-v3"
# 注意：查看效果时开启 render_mode="human"
env = gym.make(env_id, render_mode="human")

# 2. 包装环境
# 必须使用 DummyVecEnv，因为 VecNormalize 只能包装向量化环境
env = DummyVecEnv([lambda: env])

# 3. 核心步骤：加载归一化统计量
# 如果你中断了，确保这个 pkl 文件已经生成。如果没生成，机器人会像无头苍蝇一样乱飞
stats_path = "./logs/sac_panda_results/vec_normalize.pkl" # 请检查你的路径
try:
    env = VecNormalize.load(stats_path, env)
    # 评估模式下：关闭统计更新，关闭奖励归一化
    env.training = False
    env.norm_reward = False
except Exception as e:
    print(f"加载归一化文件失败，机器人可能表现异常: {e}")

# 4. 加载模型
# 指向你中断前保存的最佳模型或最新模型
# 路径示例：./logs/sac_panda_results/best_model.zip 或 sac_model_100000_steps.zip
model_path = "./logs/sac_panda_results/best_model.zip" 
model = SAC.load(model_path)

# 5. 循环运行并渲染
print("开始演示...")
obs = env.reset()
for i in range(1000):
    # deterministic=True 表示使用确定性策略（不带噪声，最强表现）
    action, _states = model.predict(obs, deterministic=True)
    obs, rewards, dones, info = env.step(action)
    
    env.render() # 弹出窗口显示
    
    if dones:
        print("回合结束，重置...")
        obs = env.reset()

env.close()

pybullet build time: Oct 21 2025 17:40:50


argv[0]=--background_color_red=0.8745098114013672
argv[1]=--background_color_green=0.21176470816135406
argv[2]=--background_color_blue=0.1764705926179886
Version = 4.1 Metal - 90.5
Vendor = Apple
Renderer = Apple M5
b3Printf: Selected demo: Physics Server
startThreads creating 1 threads.
starting thread 0
started thread 0 
MotionThreadFunc thread started
加载归一化文件失败，机器人可能表现异常: [Errno 2] No such file or directory: './logs/sac_panda_results/vec_normalize.pkl'
开始演示...
回合结束，重置...
回合结束，重置...
回合结束，重置...
回合结束，重置...
回合结束，重置...
回合结束，重置...
回合结束，重置...
回合结束，重置...
回合结束，重置...
回合结束，重置...
回合结束，重置...
回合结束，重置...
回合结束，重置...
回合结束，重置...
回合结束，重置...
回合结束，重置...
回合结束，重置...
回合结束，重置...
回合结束，重置...
回合结束，重置...
numActiveThreads = 0
stopping threads
Thread with taskId 0 exiting
Thread TERMINATED
destroy semaphore
semaphore destroyed
destroy main semaphore
main semaphore destroyed
